# Ontology 101, Part 1: What an Ontology Actually Is

Companion notebook for the article. It builds a small OWL ontology for a B2B SaaS domain with rdflib, writes the churn rule as an OWL axiom, queries the graph with SPARQL, and saves a `.ttl` file you will reuse in Part 4 (OntoBricks on Databricks).

Runs anywhere Python runs, including a Databricks notebook. No Databricks features are needed for Part 1. SPARQL here matches what you asserted. It does not run OWL. Load the saved file into Protégé to see the reasoner classify Initech as Churned.

In [ ]:
%pip install rdflib

## 1. The model, decided on paper first

Things: **Account**, **Subscription**, **Plan** (Free, Pro, Enterprise), **User**, **SupportTicket**.

Relationships: an Account **has** Subscriptions, a Subscription **is on** at most one Plan, a User **belongs to** at most one Account, a SupportTicket **is filed by** one User.

Rules, in the file, not only in a query: Pro and Enterprise are kinds of **PaidPlan**, disjoint from **FreePlan**. An Account with no active subscription is **Churned** (definition A; the competing usage-based definition B arrives with usage events in Part 2).

Everything below is transcription of those sentences.

## 2. Classes and taxonomy

In [2]:
from rdflib import Graph, Namespace, Literal, RDF, RDFS, OWL, XSD

SAAS = Namespace("http://example.org/saas#")
g = Graph()
g.bind("saas", SAAS)
g.bind("owl", OWL)

# Classes: the kinds of things that exist
for cls in ["Account", "Subscription", "Plan", "User", "SupportTicket"]:
    g.add((SAAS[cls], RDF.type, OWL.Class))

# Taxonomy: paid plans are plans
g.add((SAAS.PaidPlan, RDF.type, OWL.Class))
g.add((SAAS.PaidPlan, RDFS.subClassOf, SAAS.Plan))
g.add((SAAS.FreePlan, RDF.type, OWL.Class))
g.add((SAAS.FreePlan, RDFS.subClassOf, SAAS.Plan))

print(f"{len(g)} triples so far")

9 triples so far


## 3. Properties

Object properties connect things to things. Datatype properties attach literal values. `rdfs:domain` and `rdfs:range` are inference axioms, not checks. If you write `Acme filedBy Sub_001`, a reasoner infers that Acme is a SupportTicket. It does not reject the triple. SHACL, in Part 4, is the check. `owl:FunctionalProperty` is the at-most-one half of the paper list.

In [3]:
# Object properties: thing-to-thing relationships
props = [
    ("hasSubscription", SAAS.Account, SAAS.Subscription),
    ("onPlan",          SAAS.Subscription, SAAS.Plan),
    ("belongsTo",       SAAS.User, SAAS.Account),
    ("filedBy",         SAAS.SupportTicket, SAAS.User),
]
for name, domain, range_ in props:
    p = SAAS[name]
    g.add((p, RDF.type, OWL.ObjectProperty))
    g.add((p, RDFS.domain, domain))
    g.add((p, RDFS.range, range_))

g.add((SAAS.onPlan, RDF.type, OWL.FunctionalProperty))
g.add((SAAS.belongsTo, RDF.type, OWL.FunctionalProperty))

# Datatype properties: attributes
g.add((SAAS.mrr, RDF.type, OWL.DatatypeProperty))
g.add((SAAS.mrr, RDFS.domain, SAAS.Subscription))
g.add((SAAS.mrr, RDFS.range, XSD.decimal))

g.add((SAAS.isActive, RDF.type, OWL.DatatypeProperty))
g.add((SAAS.isActive, RDFS.domain, SAAS.Subscription))
g.add((SAAS.isActive, RDFS.range, XSD.boolean))

print(f"{len(g)} triples so far")

29 triples so far


## 4. Instances

One healthy account (Acme, Enterprise, active) and one churned account (Initech, Pro, lapsed).

In [4]:
# The three plans. Free is a FreePlan. Pro and Enterprise are PaidPlans.
g.add((SAAS.Free, RDF.type, SAAS.FreePlan))
g.add((SAAS.Pro, RDF.type, SAAS.PaidPlan))
g.add((SAAS.Enterprise, RDF.type, SAAS.PaidPlan))

# One account with a live subscription
g.add((SAAS.Acme, RDF.type, SAAS.Account))
g.add((SAAS.Sub_001, RDF.type, SAAS.Subscription))
g.add((SAAS.Acme, SAAS.hasSubscription, SAAS.Sub_001))
g.add((SAAS.Sub_001, SAAS.onPlan, SAAS.Enterprise))
g.add((SAAS.Sub_001, SAAS.mrr, Literal(4200.00, datatype=XSD.decimal)))
g.add((SAAS.Sub_001, SAAS.isActive, Literal(True)))

# And one that churned: subscription exists, but lapsed
g.add((SAAS.Initech, RDF.type, SAAS.Account))
g.add((SAAS.Sub_002, RDF.type, SAAS.Subscription))
g.add((SAAS.Initech, SAAS.hasSubscription, SAAS.Sub_002))
g.add((SAAS.Sub_002, SAAS.onPlan, SAAS.Pro))
g.add((SAAS.Sub_002, SAAS.mrr, Literal(900.00, datatype=XSD.decimal)))
g.add((SAAS.Sub_002, SAAS.isActive, Literal(False)))

print(f"{len(g)} triples so far")

44 triples so far


## 5. The churn rule, as OWL

`ActiveSubscription` is a subscription whose `isActive` value is true. `Churned` is an account that has no such subscription. `PaidPlan` and `FreePlan` cannot overlap. Part 4's reasoner will fire these axioms. SPARQL in the next cells will not.

In [5]:
g.parse(data="""
@prefix saas: <http://example.org/saas#> .
@prefix owl:  <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .

saas:PaidPlan owl:disjointWith saas:FreePlan .

saas:ActiveSubscription a owl:Class ;
    owl:equivalentClass [
        a owl:Class ;
        owl:intersectionOf (
            saas:Subscription
            [
                a owl:Restriction ;
                owl:onProperty saas:isActive ;
                owl:hasValue true
            ]
        )
    ] .

saas:Churned a owl:Class ;
    rdfs:subClassOf saas:Account ;
    owl:equivalentClass [
        a owl:Class ;
        owl:intersectionOf (
            saas:Account
            [
                a owl:Class ;
                owl:complementOf [
                    a owl:Restriction ;
                    owl:onProperty saas:hasSubscription ;
                    owl:someValuesFrom saas:ActiveSubscription
                ]
            ]
        )
    ] .
""", format="turtle")

print(f"{len(g)} triples total")

70 triples total


## 6. Query: total MRR on paid plans

The query never names Pro or Enterprise. `rdfs:subClassOf*` walks the taxonomy, so a Premium plan added next quarter is covered the moment it is declared a PaidPlan.

In [6]:
q = """
PREFIX saas: <http://example.org/saas#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT (SUM(?mrr) AS ?activePaidMrr)
WHERE {
  ?account saas:hasSubscription ?sub .
  ?sub saas:onPlan ?plan ;
       saas:mrr ?mrr ;
       saas:isActive true .
  ?plan a ?planType .
  ?planType rdfs:subClassOf* saas:PaidPlan .
}
"""
for row in g.query(q):
    print("Active paid MRR:", row.activePaidMrr)   # 4200.0

Active paid MRR: 4200.0


## 7. Query: churn, definition A

This SPARQL restates the condition so you can check the two accounts without a reasoner. The OWL axiom is what Protégé and Part 4 will apply. The next cell asks `?account a saas:Churned`. It returns nothing, because rdflib does not run OWL.

In [7]:
q_churn = """
PREFIX saas: <http://example.org/saas#>

SELECT ?account
WHERE {
  ?account a saas:Account .
  FILTER NOT EXISTS {
    ?account saas:hasSubscription ?sub .
    ?sub saas:isActive true .
  }
}
"""
for row in g.query(q_churn):
    print("Churned (definition A, SPARQL):", row.account)   # http://example.org/saas#Initech

q_typed = """
PREFIX saas: <http://example.org/saas#>
SELECT ?account WHERE { ?account a saas:Churned }
"""
print("Typed as Churned without a reasoner:", list(g.query(q_typed)))  # []

Churned (definition A, SPARQL): http://example.org/saas#Initech
Typed as Churned without a reasoner: []


## 8. Save the artifact

Keep `saas_ontology.ttl`. Confirm `owl:equivalentClass` for `Churned` and `owl:disjointWith` for the plan classes are in the file. In Part 4 the file imports into OntoBricks and gets mapped onto Unity Catalog tables. If the file has no axioms, that reasoner has nothing to do.

In [8]:
g.serialize(destination="saas_ontology.ttl", format="turtle")
ttl = open("saas_ontology.ttl").read()
assert "owl:equivalentClass" in ttl or "equivalentClass" in ttl
assert "disjointWith" in ttl
print(ttl[:1500])

@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix saas: <http://example.org/saas#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

saas:Acme a saas:Account ;
    saas:hasSubscription saas:Sub_001 .

saas:Churned a owl:Class ;
    rdfs:subClassOf saas:Account ;
    owl:equivalentClass [ a owl:Class ;
            owl:intersectionOf ( saas:Account [ a owl:Class ;
                        owl:complementOf [ a owl:Restriction ;
                                owl:onProperty saas:hasSubscription ;
                                owl:someValuesFrom saas:ActiveSubscription ] ] ) ] .

saas:Free a saas:FreePlan .

saas:Initech a saas:Account ;
    saas:hasSubscription saas:Sub_002 .

saas:belongsTo a owl:FunctionalProperty,
        owl:ObjectProperty ;
    rdfs:domain saas:User ;
    rdfs:range saas:Account .

saas:filedBy a owl:ObjectProperty ;
    rdfs:domain saas:Sup

## 9. Verify

In [9]:
results_mrr = list(g.query(q))
results_churn = list(g.query(q_churn))
assert float(results_mrr[0].activePaidMrr) == 4200.0, "MRR query returned an unexpected value"
assert len(results_churn) == 1 and str(results_churn[0].account).endswith("Initech"), "Churn query returned an unexpected account"
print("All checks passed. Keep saas_ontology.ttl for Part 4. Open it in Protégé and run the reasoner to see Initech under Churned.")

All checks passed. Keep saas_ontology.ttl for Part 4. Open it in Protégé and run the reasoner to see Initech under Churned.


## 10. Protégé

Load `saas_ontology.ttl` into [Protégé](https://protege.stanford.edu/) and start HermiT or Pellet.

1. Look up Initech. You asserted it as an Account. After reasoning it should also appear under Churned.
2. Assert that Free is also a PaidPlan, re-run, and the reasoner should flag an inconsistency because PaidPlan and FreePlan are disjoint.

If Initech does not show up under Churned, the axiom did not load. Confirm `saas:Churned` and `owl:equivalentClass` are in the Turtle, then reload.

**Next: Part 2, Knowledge graphs and GraphRAG.** Bring the `.ttl` file.